In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import pandas as pd

from icicle.analysis.bond_breaking import (
    compute_bond_scores,
    extract_bond_breaking_events,
    load_results_from_hdf5,
    plot_scores_heatmap,
)

# Random split — full test set (max_molecules=None = all ~3018 molecules)
HDF5_PATH_RDM = "/home/magled/icicle-dev/results/eval/entropy_scaffold_s2_sim/all_evaluation_spectra.hdf5"
METADATA_PATH_NIST = (
    "/home/magled/icicle-dev/data/NIST2023_GCMS_main/metadata.tsv"
)

results_rdm = load_results_from_hdf5(
    hdf5_path=HDF5_PATH_RDM,
    metadata_path=METADATA_PATH_NIST,
    max_molecules=None,
    threshold=0.01,
    max_nodes=100,
)
print(f"Loaded {len(results_rdm)} molecules from random-split HDF5.")

events_rdm = pd.concat(
    [extract_bond_breaking_events(r) for r in results_rdm.values()],
    ignore_index=True,
)
scores_rdm = compute_bond_scores(events_rdm, results_rdm)
print("P-scores (random split):")
p_rdm = scores_rdm["preferential_score"]
for bond in sorted(p_rdm, key=p_rdm.get, reverse=True):
    print(f"  {bond:10s}  P={p_rdm[bond]:5.1f}%")
fig_rdm = plot_scores_heatmap(scores_rdm, row_normalize=True)

In [ ]:
fig_rdm_10 = plot_scores_heatmap(scores_rdm, row_normalize=True, top_n=10)

In [ ]:
fig_rdm_20 = plot_scores_heatmap(scores_rdm, row_normalize=True, top_n=20)

In [ ]:
counts = scores_rdm["absolute_counts"]
top10 = sorted(counts, key=counts.get, reverse=True)[:10]
top12 = sorted(counts, key=counts.get, reverse=True)[:12]
top15 = sorted(counts, key=counts.get, reverse=True)[:15]
top20 = sorted(counts, key=counts.get, reverse=True)[:20]

metrics = ["preferential_score", "intensity_score", "absolute_counts"]

fig_rdm = plot_scores_heatmap(
    scores_rdm, metrics=metrics, row_normalize=True, bond_subset=top10
)

fig_rdm_12 = plot_scores_heatmap(
    scores_rdm, row_normalize=True, bond_subset=top12, metrics=metrics
)
fig_rdm_20 = plot_scores_heatmap(
    scores_rdm,
    row_normalize=True,
    bond_subset=top20,
    metrics=metrics,
    save_path="/home/magled/icicle-dev/results/eval/bond_scores_heatmap_top20.png",
)


fig_rdm = plot_scores_heatmap(scores_rdm, row_normalize=True, metrics=metrics)

### Zeroth coordination sphere — full random test set

Bond-order agnostic (C~N collapses C-N, C=N, C#N). Answers: which atom-type pairs are intrinsically fragile across 2500+ molecules?

In [ ]:
from icicle.analysis.bond_breaking import (
    aggregate_zeroth_patterns,
    compute_zeroth_scores,
    plot_scores_heatmap,
    plot_zeroth_breaking_stats,
)

agg_zeroth_rdm = aggregate_zeroth_patterns(events_rdm)
print(agg_zeroth_rdm.to_string(index=False))

fig_zeroth_rdm = plot_zeroth_breaking_stats(agg_zeroth_rdm, top_n=20)

scores_zeroth_rdm = compute_zeroth_scores(events_rdm, results_rdm)
p0_rdm = scores_zeroth_rdm["preferential_score"]
occ0_rdm = scores_zeroth_rdm["bond_occurrences"]
print("\nP-score (zeroth sphere, random test set):")
for bond in sorted(p0_rdm, key=p0_rdm.get, reverse=True):
    print(
        f"  {bond:8s}  P={p0_rdm[bond]:5.1f}%  occurrences={occ0_rdm.get(bond, 0)}"
    )

fig_scores_zeroth_rdm = plot_scores_heatmap(
    scores_zeroth_rdm, row_normalize=True
)

### Second coordination sphere — full random test set

Each bond-end atom's local environment includes its other heavy-atom neighbors (e.g. `C.sp3(Cl,Br)`). Top environments printed as SMARTS strings.

In [ ]:
from icicle.analysis.bond_breaking import (
    compute_sphere2_scores,
    enrich_bond_events,
    enrich_with_second_sphere,
    plot_scores_heatmap,
    plot_second_sphere_heatmap,
    plot_smarts_gallery,
)

enriched_rdm = enrich_bond_events(events_rdm, results_rdm)
sphere2_rdm = enrich_with_second_sphere(enriched_rdm, results_rdm)

# SMARTS text table — top environments by count
top_envs = (
    sphere2_rdm.dropna(subset=["bond_env_key"])
    .groupby("bond_env_key")
    .agg(count=("intensity", "count"), total_intensity=("intensity", "sum"))
    .sort_values("count", ascending=False)
    .head(30)
)
print(
    "Top second-sphere environments (SMARTS / label  |  count  |  total_intensity):"
)
for key, row in top_envs.iterrows():
    print(
        f"  {key:<50s}  count={int(row['count']):5d}  intensity={row['total_intensity']:.2f}"
    )

# Heatmap views
fig_s2_rdm_count = plot_second_sphere_heatmap(
    sphere2_rdm, metric="count", top_n_envs=15
)
fig_s2_rdm_intensity = plot_second_sphere_heatmap(
    sphere2_rdm, metric="total_intensity", top_n_envs=15
)
fig_s2_rdm_intensity = plot_second_sphere_heatmap(
    sphere2_rdm, metric="p_score", top_n_envs=15
)

# P-score
scores_s2_rdm = compute_sphere2_scores(sphere2_rdm, results_rdm)
p2_rdm = scores_s2_rdm["preferential_score"]
occ2_rdm = scores_s2_rdm["bond_occurrences"]
print("\nP-score (second sphere, top 20):")
for pair in sorted(p2_rdm, key=p2_rdm.get, reverse=True)[:20]:
    print(
        f"  {pair}  P={p2_rdm[pair]:.1f}%  occurrences={occ2_rdm.get(pair, 0)}"
    )

fig_scores_s2_rdm = plot_scores_heatmap(scores_s2_rdm, row_normalize=True)

# SMARTS gallery
fig_gallery_rdm = plot_smarts_gallery(
    sphere2_rdm, results_rdm, top_n=16, cols=4
)

In [ ]:
from icicle.analysis.bond_breaking import plot_bond_environment_heatmap

# 1st-order heatmap by absolute count — top 10 and top 20 bond types (random all sample)
fig_rdm_1st_count_10 = plot_bond_environment_heatmap(
    enriched_rdm, metric="count", top_n_envs=10
)
fig_rdm_1st_count_20 = plot_bond_environment_heatmap(
    enriched_rdm, metric="count", top_n_envs=20
)

In [ ]:
from icicle.analysis.bond_breaking import plot_bond_environment_heatmap

# 1st-order heatmap by absolute count — top 10 and top 20 bond types (random all sample)
fig_rdm_1st_count_10 = plot_bond_environment_heatmap(
    enriched_rdm, metric="intensity", top_n_envs=10
)
fig_rdm_1st_count_20 = plot_bond_environment_heatmap(
    enriched_rdm, metric="intensity", top_n_envs=20
)

In [ ]:
from icicle.analysis.bond_breaking import plot_bond_environment_heatmap

# 1st-order heatmap by absolute count — top 10 and top 20 bond types (random all sample)
fig_rdm_1st_count_10 = plot_bond_environment_heatmap(
    enriched_rdm, metric="p_score", top_n_envs=10
)
fig_rdm_1st_count_20 = plot_bond_environment_heatmap(
    enriched_rdm, metric="p_score", top_n_envs=20
)